In [ ]:
import pandas as pd

# Path to data frame with WSIs
# df_path = r"D:\DATA\with_snomed_category.csv"
# df_path = r"D:\DATA\abmil_exp2.csv"
# df_path = r"D:\DATA\abmil_exp3.csv"
df_path = r"D:\DATA\abmil_inference_exp3.csv"

df_all = pd.read_csv(df_path)
print(df_all.columns)

# Paths for ABMIL inference output
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"
checkpoint_path = r"D:\NOTEBOOKS\Christine\checkpoints\exp3_h-optimus-0\fold_1_auc_0.8883.pt"
cache_path = r"D:\NOTEBOOKS\Christine\checkpoints\exp3_h-optimus-0\fold_1_inference_cache.pkl"

In [ ]:
from abmil_pipeline import DiseaseClassification

all_filenames = df_all['filename'].tolist()

classifier = DiseaseClassification(checkpoint_path=checkpoint_path, zarr_dir=zarr_dir, slides=all_filenames, cache_path = cache_path)
print('Classifier initialized for', len(all_filenames), 'slides')

In [ ]:
# Prepare true labels if available
true_labels = df_all.get('M_idx', None)
if true_labels is not None:
    true_labels = true_labels.tolist()

classifier.process_slides(true_labels=true_labels, top_k=100)

slide_cache = classifier._slide_cache

In [ ]:
# Get assessment report + confusion matrix  
report = classifier.assessment_report(true_labels=true_labels)

In [ ]:
# Get top tiles as ROIs
slide = all_filenames[0]  # Change index to select different slide
slide_data = slide_cache[slide]
top_tiles_df = slide_data["top_tiles_df"].copy()

In [ ]:
import spatialdata
import spatialdata_plot 
from napari_spatialdata import Interactive 
from dvpio.read.image import read_openslide
from dvpio.read.shapes import read_lmd

In [ ]:
# Initialize SpatialData object
sdata = spatialdata.SpatialData()
sdata

In [ ]:
img = read_openslide(slide)
img

In [ ]:
sdata.images["image"] = img
sdata

sdata.shapes["tiles"] = ShapesModel.parse(tiles_gdf)

In [ ]:
fullres = sdata.images["image"]

lowres = fullres.coarsen(
    {fullres.dims[-2]: display_downsample, fullres.dims[-1]: display_downsample},
    boundary="trim",
).mean()

sdata.images["image_vis"] = Image2DModel.parse(lowres)

import random
from shapely.geometry import box
import geopandas as gpd
from spatialdata.models import ShapesModel
import numpy as np

H = img.shape[1]  # image height
W = img.shape[2]  # image width
H = H // display_downsample
W = W // display_downsample

tile_size = 224 // display_downsample  # tile size in low-res coordinates
n_tiles = 20

tiles = []

# your sampling region
x_min, x_max = int(H * 0.3), int(H * 0.7)
y_min, y_max = int(W * 0.3), int(W * 0.7)

max_attempts = 10000
attempts = 0

while len(tiles) < n_tiles and attempts < max_attempts:
    attempts += 1

    x = random.randint(x_min, x_max - tile_size)
    y = random.randint(y_min, y_max - tile_size)

    new_tile = box(x, y, x + tile_size, y + tile_size)

    # check overlap
    if any(new_tile.intersects(existing) for existing in tiles):
        continue

    tiles.append(new_tile)

print(f"Generated {len(tiles)} non-overlapping tiles in {attempts} attempts")

tiles_gdf = gpd.GeoDataFrame(
    {"tile_id": [f"tile_{i}" for i in range(len(tiles))]},
    geometry=tiles,
)

sdata.shapes["tiles"] = ShapesModel.parse(tiles_gdf)

polygons = [
    np.array(g.exterior.coords)
    for g in tiles_gdf.geometry
    if g.geom_type == "Polygon"
]

In [ ]:
# In the next cell, open Napari, add a `calibration_points` points layer and a `polygons` shape layer, then close the viewer.

import geopandas as gpd
import numpy as np
import spatialdata as sd
from shapely.geometry import Polygon
from spatialdata.models import ShapesModel




roi_polygons = [
    np.array(geometry.exterior.coords)
    for geometry in tiles_gdf.geometry
    if geometry.geom_type == "Polygon"
]

In [ ]:
import napari

viewer = napari.Viewer()
viewer.add_shapes(
    roi_polygons,
    shape_type="polygon",
    edge_color="red",
    face_color="red",
    name="top_tiles_roi",
)
viewer.add_points(np.empty((0, 2)), name="calibration_points")
viewer.add_shapes([], shape_type="polygon", name="polygons")

# Use the ROI layer for reference, then draw the calibration points and polygons in Napari.
napari.run()

In [ ]:
from spatialdata.models import PointsModel

# After closing Napari, collect the manually created layers.
points_layer = viewer.layers["calibration_points"]
image_points = points_layer.data

if len(image_points) == 0:
    raise ValueError("Add at least one point to the calibration_points layer before continuing.")

sdata.points["calibration_points"] = PointsModel.parse(np.array(image_points))

square_layer = viewer.layers["polygons"]
image_square = square_layer.data

if len(image_square) == 0:
    raise ValueError("Add at least one polygon to the polygons layer before continuing.")

square_polygons = [Polygon(coords) for coords in image_square]
square_gdf = gpd.GeoDataFrame(
    {"shape_id": [f"square_{i}" for i in range(len(square_polygons))]},
    geometry=square_polygons,
)
sdata.shapes["square"] = ShapesModel.parse(square_gdf)

In [ ]:
import os

from dvpio.write import write_lmd

# Convert the manually created annotations into an LMD XML file.
slide_height = int(slide_data["tile_table"]["geometry"].bounds["maxy"].max()) + 1
path_lmd = os.path.join(Path(cache_path).parent, f"{Path(slide_path).stem}_top_tiles.xml")

affine_transformation = np.array([
    [1,  0, 0],
    [0, -1, slide_height],
    [0,  0, 1],
])

write_lmd(
    path_lmd,
    sdata.shapes["tiles"],
    calibration_points=sdata.points["calibration_points"],
    affine_transformation=affine_transformation,
)

print(f"Saved LMD file to {path_lmd}")